# 🧠 RAG Master Cheatsheet
> **Stack**: Groq API + Local Vector DB + PDF Ingestion  
> **Level**: Advanced — Production Patterns, Optimizations, Evaluation

---

## 📋 Table of Contents
1. [Core Concepts & Pipeline Overview](#1)
2. [PDF Ingestion & Chunking Strategies](#2)
3. [Embeddings](#3)
4. [Local Vector DB (FAISS / ChromaDB)](#4)
5. [Retrieval Strategies](#5)
6. [Groq API — LLM Generation](#6)
7. [Advanced RAG Patterns](#7)
8. [Evaluation Metrics](#8)
9. [Production Optimizations](#9)
10. [Common Failure Modes & Fixes](#10)

---

<a id='1'></a>
## 1. Core Concepts & Pipeline Overview

RAG = **Retrieve** relevant context → **Augment** the prompt → **Generate** an answer.

```
PDF  ──► Chunks ──► Embeddings ──► Vector DB
                                       │
Query ──► Embed ──► Similarity Search ─┘
                          │
                    Top-K Chunks
                          │
                   Prompt + Context
                          │
                     Groq LLM
                          │
                       Answer
```

| Term | Meaning |
|---|---|
| **Chunk** | Small text segment from your doc |
| **Embedding** | Dense vector representing meaning |
| **Vector DB** | DB that stores & searches vectors |
| **Retriever** | Component that fetches top-K chunks |
| **Context window** | Max tokens the LLM can see |
| **Grounding** | Anchoring answer in retrieved facts |

In [ ]:
# ─── Install dependencies ───────────────────────────────────────────────────
# Run this once in your environment

# !pip install groq langchain langchain-community faiss-cpu chromadb
# !pip install pypdf sentence-transformers tiktoken rank_bm25
# !pip install ragas tqdm

<a id='2'></a>
## 2. PDF Ingestion & Chunking Strategies

Chunking is the **most impactful** decision in RAG. Wrong chunk size = bad retrieval.

| Strategy | Best For | Chunk Size |
|---|---|---|
| Fixed-size | Simple docs, fast baseline | 512–1024 tokens |
| Recursive split | General purpose ✅ | 500 + 50 overlap |
| Semantic split | High accuracy, slower | Variable |
| Sentence-level | QA over precise facts | 1–3 sentences |
| Sliding window | Dense technical docs | 512 + 128 overlap |

In [ ]:
# ─── 2.1 Load PDF ───────────────────────────────────────────────────────────

from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader("your_document.pdf")
pages = loader.load()  # Returns list of Document objects, one per page

print(f"Total pages: {len(pages)}")
print(f"First page preview: {pages[0].page_content[:200]}")
# Output → 'Introduction This paper describes...'

In [ ]:
# ─── 2.2 Chunking Strategies ────────────────────────────────────────────────

from langchain.text_splitter import RecursiveCharacterTextSplitter

# ✅ RECOMMENDED: RecursiveCharacterTextSplitter
# Tries to split by paragraphs → sentences → words (respects structure)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # ~500 chars per chunk  
    chunk_overlap=50,     # 50 char overlap to avoid cutting context mid-thought
    separators=["\n\n", "\n", ".", " ", ""]  # priority order
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")
print(f"Example chunk:\n{chunks[5].page_content}")
# Output → 'The transformer architecture introduced in 2017...'

In [ ]:
# ─── 2.3 Add Metadata to Chunks (CRITICAL for production) ───────────────────
# Metadata lets you filter retrieval later (e.g., only search chapter 3)

for i, chunk in enumerate(chunks):
    chunk.metadata.update({
        "chunk_id": i,
        "source": "your_document.pdf",
        "page": chunk.metadata.get("page", 0),
        "char_count": len(chunk.page_content)
    })

# Example metadata output:
# {'chunk_id': 5, 'source': 'paper.pdf', 'page': 2, 'char_count': 487}

In [ ]:
# ─── 2.4 Pro Tips: Token-based chunking (more accurate than char-based) ─────

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # works for most modern models

def count_tokens(text):
    return len(enc.encode(text))

# Check your chunks aren't too big
token_counts = [count_tokens(c.page_content) for c in chunks]
print(f"Max tokens in a chunk: {max(token_counts)}")
print(f"Avg tokens per chunk:  {sum(token_counts)//len(token_counts)}")
# ⚠️  Keep chunks < 512 tokens for most embedding models

<a id='3'></a>
## 3. Embeddings

Embeddings convert text → vectors so we can do **semantic similarity search**.

| Model | Dims | Speed | Best For |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | ⚡ Fast | Baseline, local |
| `all-mpnet-base-v2` | 768 | Medium | Better quality |
| `BAAI/bge-small-en` | 384 | ⚡ Fast | SOTA small model |
| `BAAI/bge-large-en` | 1024 | Slow | Best quality local |
| `text-embedding-3-small` | 1536 | API | OpenAI (paid) |

In [ ]:
# ─── 3.1 Local Embeddings with SentenceTransformers ─────────────────────────

from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",   # ✅ Best free small model
    model_kwargs={"device": "cpu"},          # Use 'cuda' if GPU available
    encode_kwargs={"normalize_embeddings": True}  # Required for cosine similarity
)

# Test it
vector = embedding_model.embed_query("What is machine learning?")
print(f"Embedding dimensions: {len(vector)}")  # Output → 384
print(f"First 5 values: {vector[:5]}")         # Output → [-0.02, 0.11, ...]

In [ ]:
# ─── 3.2 Understanding Cosine Similarity ────────────────────────────────────
# This is how the vector DB finds relevant chunks

import numpy as np

def cosine_similarity(vec1, vec2):
    """Score = 1.0 means identical, 0.0 means completely different"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Example:
v1 = embedding_model.embed_query("What is deep learning?")
v2 = embedding_model.embed_query("Explain neural networks")     # Similar topic
v3 = embedding_model.embed_query("What is the weather today?")  # Unrelated

print(f"Deep learning vs Neural nets : {cosine_similarity(v1, v2):.3f}")  # ~0.85
print(f"Deep learning vs Weather     : {cosine_similarity(v1, v3):.3f}")  # ~0.20

<a id='4'></a>
## 4. Local Vector DB — FAISS & ChromaDB

| | **FAISS** | **ChromaDB** |
|---|---|---|
| Persistence | Manual (save/load) | Auto |
| Metadata filtering | ❌ Limited | ✅ Built-in |
| Speed | ⚡ Fastest | Fast |
| Best for | Pure vector search | Full RAG pipeline |

In [ ]:
# ─── 4.1 FAISS — Build & Save ───────────────────────────────────────────────

from langchain.vectorstores import FAISS

# Build index from chunks (embeds all chunks automatically)
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# ✅ Save to disk (so you don't re-embed every run)
vectorstore.save_local("faiss_index")
print("Index saved!")

# ✅ Load from disk
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True  # Required flag in newer langchain
)
print(f"Index loaded with {vectorstore.index.ntotal} vectors")

In [ ]:
# ─── 4.2 ChromaDB — Persistent Local DB ─────────────────────────────────────

import chromadb
from langchain.vectorstores import Chroma

# Persists to ./chroma_db folder on disk
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db",
    collection_name="my_pdf_collection"
)
vectorstore.persist()  # Flush to disk

# ✅ Load existing collection
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model,
    collection_name="my_pdf_collection"
)

# Metadata filtering — very useful!
results = vectorstore.similarity_search(
    query="transformer architecture",
    k=3,
    filter={"page": 2}  # Only search page 2
)

<a id='5'></a>
## 5. Retrieval Strategies

**Retrieval quality = answer quality.** Different strategies for different needs.

| Strategy | How it works | Use when |
|---|---|---|
| **Dense** | Pure vector similarity | General semantic queries |
| **BM25 (sparse)** | Keyword matching | Exact terms, names, codes |
| **Hybrid** | Dense + Sparse combined | Best of both ✅ |
| **MMR** | Max marginal relevance | Avoid duplicate chunks |
| **Multi-query** | Multiple query rewriting | Vague questions |

In [ ]:
# ─── 5.1 Basic Dense Retrieval ──────────────────────────────────────────────

query = "How does attention mechanism work?"

# Simple similarity search
docs = vectorstore.similarity_search(query, k=4)

# With scores (lower = more similar in FAISS L2, higher = better in cosine)
docs_with_scores = vectorstore.similarity_search_with_score(query, k=4)
for doc, score in docs_with_scores:
    print(f"Score: {score:.4f} | {doc.page_content[:80]}...")

In [ ]:
# ─── 5.2 MMR — Max Marginal Relevance ───────────────────────────────────────
# Balances relevance AND diversity. Avoids returning 4 near-identical chunks.

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,          # Return 4 chunks
        "fetch_k": 20,   # Fetch 20, then pick the most diverse 4
        "lambda_mult": 0.7  # 1.0 = pure relevance, 0.0 = pure diversity
    }
)

results = retriever.get_relevant_documents("What is attention?")
# ✅ Returns diverse, non-redundant chunks

In [ ]:
# ─── 5.3 Hybrid Retrieval (BM25 + Dense) ────────────────────────────────────
# BM25 catches exact keywords that semantic search misses (e.g., model names, IDs)

from rank_bm25 import BM25Okapi
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# BM25 sparse retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# Dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# ✅ Combine them with equal weights
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]  # Adjust: 0.3/0.7 if semantic matters more
)

results = hybrid_retriever.get_relevant_documents("BERT model architecture")
print(f"Retrieved {len(results)} chunks")

In [ ]:
# ─── 5.4 Multi-Query Retrieval ───────────────────────────────────────────────
# LLM rewrites query in 3 different ways → retrieves for all → merges results
# Great when users ask vague or short questions

from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-8b-8192", api_key="YOUR_GROQ_KEY")

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Query: "explain it" → LLM expands to:
# 1. "What is the main concept explained in the document?"
# 2. "How does the described method work?"
# 3. "What are the key ideas presented?"
results = multi_retriever.get_relevant_documents("explain it")

<a id='6'></a>
## 6. Groq API — LLM Generation

Groq runs LLMs on custom LPU hardware — much faster than GPU inference.

| Model | Context | Speed | Best For |
|---|---|---|---|
| `llama3-8b-8192` | 8K | ⚡⚡⚡ | Fast QA |
| `llama3-70b-8192` | 8K | ⚡⚡ | Better reasoning |
| `mixtral-8x7b-32768` | 32K | ⚡⚡ | Long documents |
| `gemma-7b-it` | 8K | ⚡⚡⚡ | Instruction tasks |

In [ ]:
# ─── 6.1 Groq Setup ─────────────────────────────────────────────────────────

import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
# ✅ Set env var: export GROQ_API_KEY="gsk_xxx"

# Simple test call
response = client.chat.completions.create(
    model="llama3-8b-8192",
    messages=[{"role": "user", "content": "Hello, who are you?"}],
    temperature=0.1,
    max_tokens=100
)
print(response.choices[0].message.content)

In [ ]:
# ─── 6.2 Groq via LangChain ─────────────────────────────────────────────────

from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

llm = ChatGroq(
    model="llama3-70b-8192",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,      # 0 = deterministic, better for factual QA
    max_tokens=1024
)

In [ ]:
# ─── 6.3 RAG Prompt Template ─────────────────────────────────────────────────
# The prompt template is critical — it tells the LLM HOW to use the context

RAG_PROMPT = """\
You are a helpful assistant that answers questions based ONLY on the provided context.
If the answer is not in the context, say "I don't have enough information to answer this."
Do NOT make up information.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

# Helper: format retrieved docs into a single context string
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source','?')} p.{d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    )

In [ ]:
# ─── 6.4 Full RAG Chain (LCEL — LangChain Expression Language) ───────────────

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Chain: question → retrieve → format → prompt → LLM → parse
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ✅ Run the full pipeline
question = "What is the main contribution of this paper?"
answer = rag_chain.invoke(question)
print(answer)

<a id='7'></a>
## 7. Advanced RAG Patterns

These patterns address the most common production failures.

In [ ]:
# ─── 7.1 Query Rewriting ─────────────────────────────────────────────────────
# Improve retrieval by rewriting the user's query before searching
# "tell me abt it" → "What are the main topics discussed in the document?"

REWRITE_PROMPT = """Rewrite the following user question to be clearer and more 
suitable for document retrieval. Return ONLY the rewritten question, nothing else.

Original: {question}
Rewritten:"""

def rewrite_query(question: str, llm) -> str:
    rewrite_chain = ChatPromptTemplate.from_template(REWRITE_PROMPT) | llm | StrOutputParser()
    return rewrite_chain.invoke({"question": question})

# Example
original = "what does section 3 say abt performance?"
rewritten = rewrite_query(original, llm)
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")
# Output → 'What performance metrics or results are described in section 3?'

In [ ]:
# ─── 7.2 Reranking ───────────────────────────────────────────────────────────
# Retrieve many chunks → rerank by relevance → pass only top ones to LLM
# Significantly improves precision

# Pattern: Retrieve k=20, rerank, pass top 4 to LLM
# Two approaches:

# Option A: Cross-encoder reranker (local, accurate)
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, docs: list, top_k: int = 4) -> list:
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

# Usage:
# raw_docs = vectorstore.similarity_search(query, k=20)  # fetch many
# reranked_docs = rerank(query, raw_docs, top_k=4)       # keep best 4
# answer = llm_chain.invoke({"context": format_docs(reranked_docs), ...})

In [ ]:
# ─── 7.3 Contextual Compression ─────────────────────────────────────────────
# Extract only the relevant PART of each chunk instead of entire chunk
# Saves tokens, reduces noise

from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 6})
)

# Before: returns full 500-char chunks
# After:  returns only the 1-2 sentences actually relevant to the question
compressed_docs = compression_retriever.get_relevant_documents(
    "What accuracy did the model achieve?"
)

In [ ]:
# ─── 7.4 Parent-Child Chunking (Hierarchical Retrieval) ─────────────────────
# Small chunks for retrieval (precise match)
# Large parent chunks sent to LLM (more context)

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryByteStore

# Child splitter: small chunks for searching
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# Parent splitter: larger chunks to return
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

docstore = InMemoryByteStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_retriever.add_documents(pages)

# Search finds small child chunk → returns large parent chunk to LLM
results = parent_retriever.get_relevant_documents("attention is all you need")
print(f"Returned chunk size: {len(results[0].page_content)} chars")  # ~1000

In [ ]:
# ─── 7.5 Conversational RAG (Multi-turn with Memory) ────────────────────────
# Condenses chat history into a standalone question for retrieval

from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory,
    return_source_documents=True  # ✅ See which chunks were used
)

# Turn 1
r1 = conv_chain({"question": "What is the paper about?"})
print(r1["answer"])

# Turn 2 — uses history to understand "it"
r2 = conv_chain({"question": "What dataset did it use?"})
print(r2["answer"])

<a id='8'></a>
## 8. Evaluation Metrics

| Metric | Measures | Score range |
|---|---|---|
| **Faithfulness** | Answer grounded in context? | 0–1 |
| **Answer Relevancy** | Answer relevant to question? | 0–1 |
| **Context Recall** | Retrieved enough relevant chunks? | 0–1 |
| **Context Precision** | Are retrieved chunks all useful? | 0–1 |
| **MRR** | Is the best chunk ranked first? | 0–1 |

In [ ]:
# ─── 8.1 RAGAS — Automated RAG Evaluation ───────────────────────────────────
# RAGAS evaluates your RAG pipeline WITHOUT needing human labels

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from datasets import Dataset

# Build evaluation dataset
eval_data = {
    "question": ["What is the paper about?", "What model was used?"],
    "answer": ["The paper is about transformers.", "They used BERT."],
    "contexts": [
        ["This paper introduces the Transformer architecture..."],
        ["We fine-tuned BERT on the downstream task..."]
    ],
    "ground_truth": [  # Optional: reference answers
        "The paper introduces Transformer architecture.",
        "The study used BERT."
    ]
}

dataset = Dataset.from_dict(eval_data)

results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print(results.to_pandas())
# Output:
# faithfulness  answer_relevancy  context_precision  context_recall
#      0.92           0.88               0.75              0.83

In [ ]:
# ─── 8.2 Manual Evaluation Helpers ──────────────────────────────────────────

def retrieval_score(query, relevant_chunk_ids, retrieved_docs):
    """
    MRR (Mean Reciprocal Rank) — measures if the best chunk is ranked first
    Score of 1.0 = perfect, 0.5 = correct chunk was ranked 2nd
    """
    retrieved_ids = [d.metadata.get("chunk_id") for d in retrieved_docs]
    for rank, cid in enumerate(retrieved_ids, 1):
        if cid in relevant_chunk_ids:
            return 1.0 / rank
    return 0.0

# Example:
# query = "What accuracy was achieved?"
# relevant_ids = [23, 24]  # chunk IDs you labeled as relevant
# retrieved_docs = retriever.get_relevant_documents(query)
# mrr = retrieval_score(query, relevant_ids, retrieved_docs)
# print(f"MRR: {mrr}")  # 0.5 means correct chunk was 2nd in results

<a id='9'></a>
## 9. Production Optimizations

Things that matter when moving from notebook → real app.

In [ ]:
# ─── 9.1 Cache Embeddings — Don't Re-embed on Every Run ─────────────────────
# Embedding 1000 chunks takes time. Cache them.

import pickle, os, hashlib

def get_or_create_vectorstore(chunks, embedding_model, cache_path="cache.pkl"):
    """
    Load from cache if exists, otherwise build + cache.
    Hash the chunk content to detect if source changed.
    """
    # Hash all chunk content to detect changes
    content_hash = hashlib.md5(
        "".join(c.page_content for c in chunks).encode()
    ).hexdigest()
    
    hash_file = cache_path + ".hash"
    
    if os.path.exists(cache_path):
        with open(hash_file, "r") as f:
            saved_hash = f.read()
        if saved_hash == content_hash:
            print("✅ Loading vectorstore from cache")
            with open(cache_path, "rb") as f:
                return pickle.load(f)
    
    print("🔄 Building vectorstore...")
    vs = FAISS.from_documents(chunks, embedding_model)
    with open(cache_path, "wb") as f:
        pickle.dump(vs, f)
    with open(hash_file, "w") as f:
        f.write(content_hash)
    return vs

In [ ]:
# ─── 9.2 Async Groq for Multiple Queries (Parallel) ─────────────────────────

import asyncio
from groq import AsyncGroq

async_client = AsyncGroq(api_key=os.environ["GROQ_API_KEY"])

async def answer_question(question: str, context: str) -> str:
    response = await async_client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {"role": "system", "content": "Answer based only on the context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQ: {question}"}
        ]
    )
    return response.choices[0].message.content

async def answer_batch(questions: list, contexts: list) -> list:
    tasks = [answer_question(q, c) for q, c in zip(questions, contexts)]
    return await asyncio.gather(*tasks)  # ⚡ All queries run in parallel!

# Usage in notebook:
# answers = await answer_batch(
#     ["Q1", "Q2", "Q3"],
#     ["context1", "context2", "context3"]
# )

In [ ]:
# ─── 9.3 Streaming Responses (UX improvement) ────────────────────────────────
# Stream tokens as they're generated → faster perceived response

def stream_answer(question: str, context: str):
    stream = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQ: {question}"}],
        stream=True  # ← Enable streaming
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)  # Print tokens as they arrive

# stream_answer("What is attention?", context_text)

In [ ]:
# ─── 9.4 Prompt Engineering Tips for RAG ────────────────────────────────────

# ✅ Good RAG system prompt:
SYSTEM_PROMPT = """
You are a precise document assistant. Rules:
1. Answer ONLY using the provided context
2. If unsure, say "The document doesn't mention this"
3. Cite the source page when possible: (p.3)
4. Be concise. Do not pad your answer.
"""

# ❌ Bad prompt (causes hallucinations):
BAD_PROMPT = "Answer the user's question helpfully."
# Problem: LLM will use its own knowledge, not your document

# ✅ Structured output prompt:
STRUCTURED_PROMPT = """
Based on the context, answer the question in this JSON format:
{{
  "answer": "...",
  "confidence": "high|medium|low",
  "source_page": number or null
}}
Context: {context}
Question: {question}
"""

<a id='10'></a>
## 10. Common Failure Modes & Fixes

| Problem | Symptom | Fix |
|---|---|---|
| **Chunks too large** | LLM ignores part of context | Reduce chunk size to 300–500 chars |
| **Chunks too small** | Missing context for answer | Increase or use parent-child |
| **Wrong chunks retrieved** | Answer is irrelevant | Try hybrid retrieval or reranking |
| **Hallucination** | Answer not in doc | Stronger system prompt + faithfulness check |
| **Slow embedding** | Indexing takes too long | Cache index, use smaller model |
| **Duplicate chunks** | Repetitive answers | MMR retrieval, dedup before indexing |
| **Multi-page tables** | Table data lost | Use table-aware parser (camelot, pdfplumber) |
| **Scanned PDFs** | Empty text extraction | Add OCR (pytesseract, unstructured) |

In [ ]:
# ─── 10.1 Hallucination Guard ────────────────────────────────────────────────
# After getting an answer, verify it's grounded in the context

FAITHFULNESS_PROMPT = """
Given the context and an answer, determine if the answer is fully supported by the context.
Reply with ONLY: "faithful" or "not_faithful"

Context: {context}
Answer: {answer}
"""

def check_faithfulness(answer: str, context: str, llm) -> bool:
    chain = ChatPromptTemplate.from_template(FAITHFULNESS_PROMPT) | llm | StrOutputParser()
    result = chain.invoke({"context": context, "answer": answer})
    return "faithful" in result.lower()

# Usage:
# context = format_docs(retrieved_docs)
# answer = rag_chain.invoke(question)
# is_faithful = check_faithfulness(answer, context, llm)
# if not is_faithful:
#     answer = "I cannot confidently answer based on the document."

In [ ]:
# ─── 10.2 Handling Tables in PDFs ───────────────────────────────────────────
# PyPDFLoader loses table structure. Use pdfplumber for tables.

import pdfplumber

def extract_tables_from_pdf(pdf_path: str) -> list:
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            page_tables = page.extract_tables()
            for table in page_tables:
                # Convert table to markdown string for embedding
                header = table[0]
                rows = table[1:]
                md_table = "| " + " | ".join(str(h) for h in header) + " |\n"
                md_table += "| " + " | ".join(["---"] * len(header)) + " |\n"
                for row in rows:
                    md_table += "| " + " | ".join(str(c) for c in row) + " |\n"
                tables.append({"page": page_num, "table": md_table})
    return tables

# tables = extract_tables_from_pdf("paper.pdf")
# print(tables[0]['table'])
# Output:
# | Model  | Accuracy | F1   |
# | ---    | ---      | ---  |
# | BERT   | 92.3     | 91.8 |

---
## ⚡ Quick Reference Card

```python
# Full minimal RAG in ~20 lines

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

# 1. Load
docs = PyPDFLoader("doc.pdf").load()

# 2. Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)

# 3. Embed + Index
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = FAISS.from_documents(chunks, embeddings)

# 4. Generate
llm = ChatGroq(model="llama3-8b-8192", api_key="YOUR_KEY")
qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())

# 5. Ask
print(qa.run("What is this document about?"))
```

---

## 🔑 Key Numbers to Remember

| Parameter | Good Default |
|---|---|
| Chunk size | 400–600 chars |
| Chunk overlap | 10–15% of chunk size |
| Top-K retrieval | 4–6 chunks |
| Temperature (QA) | 0 or 0.1 |
| Max context to LLM | < 60% of model's context window |
| Embed model dims | 384 (fast) or 768 (accurate) |

---
*Built with: Groq API · FAISS/ChromaDB · LangChain · HuggingFace Embeddings*